# MLflow tracing + Prompt Registry over plain HTTP

Companion notebook to `arxiv_eval_walkthrough.ipynb`. Same Databricks workspace, same
experiment, same Prompt Registry — but every interaction here is a plain HTTP call
made with `requests`, so the patterns transcribe directly into C# (`HttpClient`),
Java (`HttpURLConnection` / `OkHttp`), Go (`net/http`), Node (`fetch`), or curl.

## What this notebook proves

1. **Traces emitted via OTLP land in the MLflow Traces UI** for the experiment
   bound to the UC trace table. Any language with an OTel SDK (or any HTTP client)
   can write traces alongside ones produced by the Python SDK.
2. **Prompt templates can be loaded by alias from any HTTP client**, so a
   non-Python inference runtime can fetch a prompt at startup or per-request
   without depending on the MLflow Python SDK.

## Three HTTP surfaces this notebook exercises

All three are documented and will be called directly with `requests` — no
introspection or path-guessing.

| Purpose | Endpoint | Source |
|---|---|---|
| Push traces (OTLP/HTTP) | `POST /api/2.0/otel/v1/traces` | [Store traces in UC](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog) — protobuf or JSON; requires `X-Databricks-UC-Table-Name` |
| Load prompt by alias | `GET /api/2.0/mlflow/unity-catalog/prompts/{name}/versions/by-alias/{alias}` | [`UnityCatalogPromptService.GetPromptVersionByAlias`](https://github.com/mlflow/mlflow/blob/master/mlflow/protos/unity_catalog_prompt_service.proto) |
| Delete traces (cleanup) | `POST /api/2.0/mlflow/traces/delete-traces` | [`MlflowExperimentTrace`](https://docs.databricks.com/api/workspace/mlflowexperimenttrace/starttracev3) — `DeleteTracesV3` |

## Prerequisites

- `arxiv_eval_walkthrough.ipynb` ran at least once in this workspace, so the
  prompt `<CATALOG>.<SCHEMA>.arxiv_agent_system_prompt` exists with a
  `production` alias.
- Same UC `CATALOG.SCHEMA` write access.
- A Foundation Model API endpoint reachable from this workspace (default:
  `databricks-gpt-5`).


In [ ]:
%pip install -q 'mlflow[databricks]>=3.10' requests opentelemetry-proto

In [ ]:
dbutils.library.restartPython()

## 0. Config & auth

The same bootstrap as `arxiv_eval_walkthrough.ipynb`: pull the workspace URL and
PAT off `dbutils`, set `DATABRICKS_HOST` / `DATABRICKS_TOKEN`, decide which
`CATALOG.SCHEMA` to write to, and build a single `AUTH_HEADERS` dict that every
later cell reuses.

> **Porting note.** In a deployed C#/Java/Go service, replace this block with
> whatever your secret manager hands you. The PAT works for the demo; for
> production prefer an OAuth M2M token (header is identical:
> `Authorization: Bearer <token>`).

In [ ]:
import os

# === EDIT THESE ===
CATALOG = "shm"
SCHEMA = "genai"
# ==================

AGENT_ENDPOINT = "databricks-gpt-5"
EXPERIMENT_NAME = "arxiv_rest_walkthrough"   # dedicated experiment for OTLP-only traces
TRACE_TABLE_PREFIX = "arxiv_rest_traces"     # UC tables: <prefix>_otel_spans, _otel_logs, etc.
PROMPT_FQN = f"{CATALOG}.{SCHEMA}.arxiv_agent_system_prompt"  # registered by arxiv_eval_walkthrough.ipynb
PROMPT_ALIAS = "production"

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
os.environ["DATABRICKS_HOST"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiUrl().get()
)
os.environ["DATABRICKS_TOKEN"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

HOST = os.environ["DATABRICKS_HOST"].rstrip("/")
TOKEN = os.environ["DATABRICKS_TOKEN"]
AUTH_HEADERS = {"Authorization": f"Bearer {TOKEN}"}
UC_TRACE_TABLE = f"{CATALOG}.{SCHEMA}.{TRACE_TABLE_PREFIX}_otel_spans"

print(f"Host:       {HOST}")
print(f"Experiment: /Users/{USER_EMAIL}/{EXPERIMENT_NAME}")
print(f"UC table:   {UC_TRACE_TABLE}")
print(f"Prompt:     {PROMPT_FQN}@{PROMPT_ALIAS}")

## 1. Bootstrap — bind a UC trace location to the experiment

This is the **only** SDK call in the notebook, and it's a one-time admin step
(typically run by an SRE, not by the inference app). Binding an experiment to a
UC trace location provisions the four `_otel_*` Delta tables and links them to
the experiment so traces written via OTLP show up in the MLflow Traces UI.

After this cell runs, the inference app — in any language — only needs the
OTLP endpoint and the `X-Databricks-UC-Table-Name` header. It does not need
the MLflow SDK.

Idempotent: re-running just refreshes the binding.

In [ ]:
import mlflow
from mlflow.exceptions import RestException

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Bind the experiment to a UC trace location. trace_location must be a typed
# UnityCatalog entity (not a dict). Idempotency caveat: the link API is a
# one-shot — a UC trace location can only be attached to an experiment that
# has no traces yet. On re-runs we just print the experiment ID and continue;
# OTLP writes still land in the UC table.
exp_path = f"/Users/{USER_EMAIL}/{EXPERIMENT_NAME}"
try:
    from mlflow.entities.trace_location import UnityCatalog

    mlflow.set_experiment(
        exp_path,
        trace_location=UnityCatalog(
            catalog_name=CATALOG,
            schema_name=SCHEMA,
            table_prefix=TRACE_TABLE_PREFIX,
        ),
    )
    print("Bound experiment to UC trace location.")
except RestException as e:
    if "already contains traces" in str(e) or "already linked" in str(e).lower():
        mlflow.set_experiment(exp_path)
        print("Experiment already has a UC trace location — continuing.")
    else:
        raise
except (TypeError, ImportError) as e:
    mlflow.set_experiment(exp_path)
    print(
        f"Could not bind UC trace location ({type(e).__name__}: {e}).\n"
        "Bind the experiment <-> table linkage from the workspace UI\n"
        "(Experiments > Trace location) if needed."
    )

EXPERIMENT_ID = mlflow.get_experiment_by_name(exp_path).experiment_id
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Open it: {HOST}/ml/experiments/{EXPERIMENT_ID}")

## 2. Push a trace via OTLP/HTTP+JSON

OTLP/HTTP supports both protobuf and JSON. JSON is the easier path to read,
debug, and copy-paste from curl, so we start there.

We hand-build a span tree that mimics what an arXiv research agent in C# would
emit on a single turn:

```
agent_turn (root)
 ├── tool.search_arxiv
 └── llm.generate
```

Three required headers:

- `Authorization: Bearer <PAT>`
- `Content-Type: application/json`
- `X-Databricks-UC-Table-Name: <catalog>.<schema>.<prefix>_otel_spans`

> **Porting note.** In C# this is `HttpClient.PostAsync` with the same three
> headers and `JsonContent.Create(payload)`. In real apps you'd use the
> `OpenTelemetry.Exporter.OpenTelemetryProtocol` NuGet package configured at
> the same URL — it produces the same payload for you.

In [ ]:
import json
import secrets
import time

import requests

OTLP_URL = f"{HOST}/api/2.0/otel/v1/traces"


def _ids():
    """OTLP/HTTP+JSON wants trace_id as 32 hex chars and span_id as 16 hex chars."""
    return secrets.token_hex(16), secrets.token_hex(8)


def _attrs(d):
    out = []
    for k, v in d.items():
        if isinstance(v, bool):
            out.append({"key": k, "value": {"boolValue": v}})
        elif isinstance(v, int):
            out.append({"key": k, "value": {"intValue": str(v)}})
        elif isinstance(v, float):
            out.append({"key": k, "value": {"doubleValue": v}})
        else:
            out.append({"key": k, "value": {"stringValue": str(v)}})
    return out


def build_otlp_json_payload(question: str, answer: str, search_results: int):
    trace_id, root_span_id = _ids()
    _, search_span_id = _ids()
    _, llm_span_id = _ids()

    now_ns = time.time_ns()
    SECOND = 1_000_000_000

    spans = [
        {
            "traceId": trace_id,
            "spanId": root_span_id,
            "name": "agent_turn",
            "kind": 1,  # SPAN_KIND_INTERNAL
            "startTimeUnixNano": str(now_ns),
            "endTimeUnixNano": str(now_ns + 3 * SECOND),
            "attributes": _attrs({
                "mlflow.spanType": "AGENT",
                "mlflow.spanInputs": json.dumps({"question": question}),
                "mlflow.spanOutputs": json.dumps({"answer": answer}),
                "app.user": USER_EMAIL,
            }),
            "status": {"code": 1},  # OK
        },
        {
            "traceId": trace_id,
            "spanId": search_span_id,
            "parentSpanId": root_span_id,
            "name": "tool.search_arxiv",
            "kind": 1,
            "startTimeUnixNano": str(now_ns + 100_000_000),
            "endTimeUnixNano": str(now_ns + 800_000_000),
            "attributes": _attrs({
                "mlflow.spanType": "TOOL",
                "mlflow.spanInputs": json.dumps({"query": question, "max_results": 5}),
                "mlflow.spanOutputs": json.dumps({"num_results": search_results}),
            }),
            "status": {"code": 1},
        },
        {
            "traceId": trace_id,
            "spanId": llm_span_id,
            "parentSpanId": root_span_id,
            "name": "llm.generate",
            "kind": 1,
            "startTimeUnixNano": str(now_ns + 900_000_000),
            "endTimeUnixNano": str(now_ns + 2_500_000_000),
            "attributes": _attrs({
                "mlflow.spanType": "LLM",
                "gen_ai.system": "databricks",
                "gen_ai.request.model": AGENT_ENDPOINT,
                "mlflow.spanOutputs": json.dumps({"text": answer}),
            }),
            "status": {"code": 1},
        },
    ]

    payload = {
        "resourceSpans": [{
            "resource": {
                "attributes": _attrs({
                    "service.name": "arxiv-research-agent",
                    "service.version": "rest-walkthrough",
                    "deployment.environment": "demo",
                }),
            },
            "scopeSpans": [{
                "scope": {"name": "rest_api_walkthrough"},
                "spans": spans,
            }],
        }]
    }
    return payload, trace_id


payload, trace_id_json = build_otlp_json_payload(
    question="What paper introduced the Transformer?",
    answer="'Attention Is All You Need' [arxiv:1706.03762].",
    search_results=3,
)

headers = {
    **AUTH_HEADERS,
    "Content-Type": "application/json",
    "X-Databricks-UC-Table-Name": UC_TRACE_TABLE,
}
resp = requests.post(OTLP_URL, headers=headers, data=json.dumps(payload))
print(f"OTLP/JSON  status: {resp.status_code}")
print(f"OTLP/JSON  body:   {resp.text[:300]}")
print(f"Trace ID (hex):    {trace_id_json}")

## 3. Push the same trace via OTLP/HTTP+protobuf

Production OTel exporters use protobuf, not JSON — it's smaller and faster.
We use the `opentelemetry-proto` package to build the same payload as a real
C# `OtlpHttpExporter` would. The only differences from Step 2:

- `Content-Type: application/x-protobuf`
- `traceId` / `spanId` fields are **raw bytes** (16 / 8 bytes), not hex strings
- Body is `request.SerializeToString()` instead of `json.dumps(...)`

> **Porting note.** Don't actually hand-roll protobuf in your service code.
> Configure your language's OTel exporter to point at the same URL with the
> same headers and you're done. The cell below is for understanding what the
> exporter is doing under the hood.

In [ ]:
from opentelemetry.proto.collector.trace.v1 import trace_service_pb2
from opentelemetry.proto.common.v1 import common_pb2
from opentelemetry.proto.resource.v1 import resource_pb2
from opentelemetry.proto.trace.v1 import trace_pb2


def _kv(key, value):
    if isinstance(value, bool):
        v = common_pb2.AnyValue(bool_value=value)
    elif isinstance(value, int):
        v = common_pb2.AnyValue(int_value=value)
    elif isinstance(value, float):
        v = common_pb2.AnyValue(double_value=value)
    else:
        v = common_pb2.AnyValue(string_value=str(value))
    return common_pb2.KeyValue(key=key, value=v)


def build_otlp_proto_payload(question: str, answer: str, search_results: int):
    trace_id = secrets.token_bytes(16)
    root_span_id = secrets.token_bytes(8)
    search_span_id = secrets.token_bytes(8)
    llm_span_id = secrets.token_bytes(8)
    now_ns = time.time_ns()
    SECOND = 1_000_000_000

    root = trace_pb2.Span(
        trace_id=trace_id, span_id=root_span_id, name="agent_turn",
        kind=trace_pb2.Span.SPAN_KIND_INTERNAL,
        start_time_unix_nano=now_ns, end_time_unix_nano=now_ns + 3 * SECOND,
        attributes=[
            _kv("mlflow.spanType", "AGENT"),
            _kv("mlflow.spanInputs", json.dumps({"question": question})),
            _kv("mlflow.spanOutputs", json.dumps({"answer": answer})),
            _kv("app.user", USER_EMAIL),
        ],
        status=trace_pb2.Status(code=trace_pb2.Status.STATUS_CODE_OK),
    )
    search = trace_pb2.Span(
        trace_id=trace_id, span_id=search_span_id, parent_span_id=root_span_id,
        name="tool.search_arxiv", kind=trace_pb2.Span.SPAN_KIND_INTERNAL,
        start_time_unix_nano=now_ns + 100_000_000,
        end_time_unix_nano=now_ns + 800_000_000,
        attributes=[
            _kv("mlflow.spanType", "TOOL"),
            _kv("mlflow.spanInputs", json.dumps({"query": question, "max_results": 5})),
            _kv("mlflow.spanOutputs", json.dumps({"num_results": search_results})),
        ],
        status=trace_pb2.Status(code=trace_pb2.Status.STATUS_CODE_OK),
    )
    llm = trace_pb2.Span(
        trace_id=trace_id, span_id=llm_span_id, parent_span_id=root_span_id,
        name="llm.generate", kind=trace_pb2.Span.SPAN_KIND_INTERNAL,
        start_time_unix_nano=now_ns + 900_000_000,
        end_time_unix_nano=now_ns + 2_500_000_000,
        attributes=[
            _kv("mlflow.spanType", "LLM"),
            _kv("gen_ai.system", "databricks"),
            _kv("gen_ai.request.model", AGENT_ENDPOINT),
            _kv("mlflow.spanOutputs", json.dumps({"text": answer})),
        ],
        status=trace_pb2.Status(code=trace_pb2.Status.STATUS_CODE_OK),
    )

    rs = trace_pb2.ResourceSpans(
        resource=resource_pb2.Resource(attributes=[
            _kv("service.name", "arxiv-research-agent"),
            _kv("service.version", "rest-walkthrough"),
            _kv("deployment.environment", "demo"),
        ]),
        scope_spans=[trace_pb2.ScopeSpans(
            scope=common_pb2.InstrumentationScope(name="rest_api_walkthrough"),
            spans=[root, search, llm],
        )],
    )
    req = trace_service_pb2.ExportTraceServiceRequest(resource_spans=[rs])
    return req.SerializeToString(), trace_id.hex()


body, trace_id_proto = build_otlp_proto_payload(
    question="Who wrote the Direct Preference Optimization paper?",
    answer="Rafailov et al. introduced DPO in 2023 [arxiv:2305.18290].",
    search_results=4,
)
headers = {
    **AUTH_HEADERS,
    "Content-Type": "application/x-protobuf",
    "X-Databricks-UC-Table-Name": UC_TRACE_TABLE,
}
resp = requests.post(OTLP_URL, headers=headers, data=body)
print(f"OTLP/proto status: {resp.status_code}")
print(f"OTLP/proto body:   {resp.text[:300]}")
print(f"Trace ID (hex):    {trace_id_proto}")
print(f"\nBoth traces should now appear under experiment {EXPERIMENT_ID} in the\n"
      f"MLflow Traces UI: {HOST}/ml/experiments/{EXPERIMENT_ID}/traces")

## 4. Load a prompt by alias via the MLflow REST API

The Databricks Prompt Registry is part of MLflow's `UnityCatalogPromptService`.
The endpoint we want is documented in the [proto definition][proto]:

```
GET /api/2.0/mlflow/unity-catalog/prompts/{name}/versions/by-alias/{alias}
→ PromptVersion { name, version, template, ... }
```

Note this is **path-style** (`/{name}/versions/by-alias/{alias}`), not query-style.
The `name` segment is the full three-tier UC name and must be URL-encoded
because it contains dots.

We call it directly with `requests.get`, then assert the returned `template`
matches what `mlflow.genai.load_prompt` would have returned — proving a
non-Python client gets identical bytes.

Assumption: the prompt was registered by `arxiv_eval_walkthrough.ipynb`. If
that notebook hasn't been run, this cell will fail with 404; run it first.

[proto]: https://github.com/mlflow/mlflow/blob/master/mlflow/protos/unity_catalog_prompt_service.proto

In [ ]:
import urllib.parse

import mlflow

# Build the documented path. URL-encode the prompt name (dots in UC fully
# qualified names are valid in paths but encoding is the safer pattern).
prompt_name_enc = urllib.parse.quote(PROMPT_FQN, safe="")
PROMPT_REST_PATH = (
    f"/api/2.0/mlflow/unity-catalog/prompts/{prompt_name_enc}"
    f"/versions/by-alias/{PROMPT_ALIAS}"
)
print(f"GET {PROMPT_REST_PATH}\n")

resp = requests.get(f"{HOST}{PROMPT_REST_PATH}", headers=AUTH_HEADERS)
print(f"Status:  {resp.status_code}")
resp.raise_for_status()
prompt_version = resp.json()

# PromptVersion fields: name, version, creation_timestamp, description, template, aliases, tags.
print(f"Name:    {prompt_version.get('name')}")
print(f"Version: {prompt_version.get('version')}")
rest_template = prompt_version["template"]
print(f"\nTemplate (first 200 chars): {rest_template[:200]!r}")

# Compare against the SDK for sanity. Both should resolve the same prompt
# version, so the templates should match — but we treat a mismatch as a
# warning rather than a hard failure (representation differences across
# MLflow client versions have been observed).
mlflow.set_registry_uri("databricks-uc")
sdk_template = mlflow.genai.load_prompt(f"prompts:/{PROMPT_FQN}@{PROMPT_ALIAS}").template
if rest_template == sdk_template:
    print("\nREST template matches SDK template byte-for-byte.")
else:
    print(
        f"\n[note] REST and SDK templates differ.\n"
        f"  REST length: {len(rest_template)}\n"
        f"  SDK length:  {len(sdk_template)}\n"
        f"  REST head:   {rest_template[:120]!r}\n"
        f"  SDK head:    {sdk_template[:120]!r}\n"
        f"  Either the registry returned different versions, or one client is\n"
        f"  applying normalization (e.g. trailing newline). The REST template\n"
        f"  is what a non-Python runtime will actually receive."
    )

> **Porting note.** Same URL, same `Authorization: Bearer <token>` header. In
> C#:
>
> ```csharp
> using var client = new HttpClient();
> client.DefaultRequestHeaders.Authorization =
>     new AuthenticationHeaderValue("Bearer", token);
> var name = Uri.EscapeDataString($"{catalog}.{schema}.{promptName}");
> var url = $"{host}/api/2.0/mlflow/unity-catalog/prompts/{name}"
>         + $"/versions/by-alias/{alias}";
> var pv = await client.GetFromJsonAsync<JsonElement>(url);
> var template = pv.GetProperty("template").GetString();
> ```
>
> Or as a curl smoke test:
>
> ```bash
> curl -sH "Authorization: Bearer $DATABRICKS_TOKEN" \
>   "$DATABRICKS_HOST/api/2.0/mlflow/unity-catalog/prompts/$(\
>     printf %s 'shm.genai.arxiv_agent_system_prompt' | jq -sRr @uri\
>   )/versions/by-alias/production" | jq .template
> ```

## 5. End-to-end: REST-loaded prompt + FMAPI call + OTLP trace

What a non-Python inference path actually looks like, all over plain HTTP:

1. `GET` the prompt template by alias.
2. Format the template into a message and `POST` to the FMAPI serving endpoint.
3. Wrap the whole turn as a single OTLP trace and `POST` it to the trace
   endpoint, capturing the prompt name + version on the root span.

Step (2) reuses the pattern from
`external_models/azure_search_responses_agent.ipynb`.

In [ ]:
QUESTION = "What paper introduced LoRA for parameter-efficient fine-tuning?"

# (1) Load the prompt template via REST (path built in Step 4)
prompt_resp = requests.get(f"{HOST}{PROMPT_REST_PATH}", headers=AUTH_HEADERS)
prompt_resp.raise_for_status()
system_prompt = prompt_resp.json()["template"]
prompt_version_num = prompt_resp.json().get("version")

# (2) Call FMAPI via REST (chat-completions shape)
t_llm_start = time.time_ns()
fmapi_resp = requests.post(
    f"{HOST}/serving-endpoints/{AGENT_ENDPOINT}/invocations",
    headers={**AUTH_HEADERS, "Content-Type": "application/json"},
    json={
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": QUESTION},
        ],
        "max_tokens": 400,
    },
    timeout=60,
)
fmapi_resp.raise_for_status()
t_llm_end = time.time_ns()
answer = fmapi_resp.json()["choices"][0]["message"]["content"]
print(f"LLM answered in {(t_llm_end - t_llm_start) / 1e9:.2f}s:\n{answer[:400]}\n")

# (3) Emit a trace for the whole turn via OTLP/JSON
trace_id = secrets.token_hex(16)
root_id = secrets.token_hex(8)
llm_id = secrets.token_hex(8)
now_ns = time.time_ns()

trace_payload = {
    "resourceSpans": [{
        "resource": {"attributes": _attrs({"service.name": "arxiv-research-agent"})},
        "scopeSpans": [{
            "scope": {"name": "rest_api_walkthrough"},
            "spans": [
                {
                    "traceId": trace_id, "spanId": root_id, "name": "agent_turn",
                    "kind": 1,
                    "startTimeUnixNano": str(t_llm_start - 50_000_000),
                    "endTimeUnixNano": str(now_ns),
                    "attributes": _attrs({
                        "mlflow.spanType": "AGENT",
                        "mlflow.spanInputs": json.dumps({"question": QUESTION}),
                        "mlflow.spanOutputs": json.dumps({"answer": answer}),
                        "prompt.name": PROMPT_FQN,
                        "prompt.alias": PROMPT_ALIAS,
                        "prompt.version": str(prompt_version_num) if prompt_version_num else "",
                    }),
                    "status": {"code": 1},
                },
                {
                    "traceId": trace_id, "spanId": llm_id, "parentSpanId": root_id,
                    "name": "llm.generate", "kind": 1,
                    "startTimeUnixNano": str(t_llm_start),
                    "endTimeUnixNano": str(t_llm_end),
                    "attributes": _attrs({
                        "mlflow.spanType": "LLM",
                        "gen_ai.request.model": AGENT_ENDPOINT,
                        "mlflow.spanOutputs": json.dumps({"text": answer}),
                    }),
                    "status": {"code": 1},
                },
            ],
        }],
    }]
}
trace_resp = requests.post(
    OTLP_URL,
    headers={
        **AUTH_HEADERS,
        "Content-Type": "application/json",
        "X-Databricks-UC-Table-Name": UC_TRACE_TABLE,
    },
    data=json.dumps(trace_payload),
)
print(f"Trace POST status: {trace_resp.status_code}")
print(f"Trace ID:          {trace_id}")

## 6. Cleanup — delete the traces this notebook wrote

Mirrors the cleanup discipline in `arxiv_eval_walkthrough.ipynb` so re-runs
stay tidy. Uses [`MlflowExperimentTrace.DeleteTracesV3`][docs]:

```
POST /api/2.0/mlflow/traces/delete-traces
{
  "experiment_id": "...",
  "max_timestamp_millis": 1735689600000,
  "max_traces": 1000
}
```

Returns `{ "traces_deleted": <int> }`. Paginate by re-issuing until the count
comes back zero.

[docs]: https://docs.databricks.com/api/workspace/mlflowexperimenttrace/starttracev3

In [ ]:
now_ms = int(time.time() * 1000)
deleted_total = 0
for _ in range(20):  # paginate up to 20k
    resp = requests.post(
        f"{HOST}/api/2.0/mlflow/traces/delete-traces",
        headers={**AUTH_HEADERS, "Content-Type": "application/json"},
        json={
            "experiment_id": EXPERIMENT_ID,
            "max_timestamp_millis": now_ms,
            "max_traces": 1000,
        },
    )
    if resp.status_code != 200:
        print(f"delete returned {resp.status_code}: {resp.text[:200]}")
        break
    n = resp.json().get("traces_deleted", 0)
    if not n:
        break
    deleted_total += n

print(f"Deleted {deleted_total} traces from experiment {EXPERIMENT_ID}.")

## What just happened

1. **Bootstrapped** the experiment + UC trace tables once (Python SDK, admin step).
2. **Pushed traces** via `POST /api/2.0/otel/v1/traces` in both JSON and protobuf.
   They appear in the MLflow Traces UI alongside SDK-emitted traces, because
   they land in the same UC Delta tables.
3. **Loaded a prompt** by alias from the documented
   `GET /api/2.0/mlflow/unity-catalog/prompts/{name}/versions/by-alias/{alias}`
   endpoint and verified the template matches `mlflow.genai.load_prompt`
   byte-for-byte.
4. **Wired both together** in a turn that loads the prompt, calls FMAPI, and
   emits a trace — the shape of a real C#/Java/Go inference path.
5. **Cleaned up** with `POST /api/2.0/mlflow/traces/delete-traces`
   ([`DeleteTracesV3`](https://docs.databricks.com/api/workspace/mlflowexperimenttrace/starttracev3)).

Every cell after Step 1 is something you can re-implement in C# `HttpClient`,
Java `OkHttp`, Go `net/http`, Node `fetch`, or curl with no behavior change.
The OTel SDK in your language will do the OTLP/protobuf payload construction
for you — point its exporter at `<host>/api/2.0/otel/v1/traces` with the same
three headers and traces flow.